#  GTA VI — Social Media Sentiment & Community Reaction (Reddit)
**Pipeline de Ingeniería de Datos & NLP (Python + Hugging Face + Power BI)**

Este cuaderno consolida el flujo completo de extracción, limpieza, análisis de sentimiento multilingüe (`ES`/`EN`), detección temática y segmentación temporal en **4 etapas** para alimentar el dashboard ejecutivo en Power BI.

### 📌 Arquitectura de las 4 Etapas
| Etapa | Ventana Temporal | Criterio de Segmentación |
| :--- | :--- | :--- |
| **1. Expectativa** | `01 Nov – 18 Nov 2026` | Publicaciones previas al lanzamiento oficial. |
| **2. Lanzamiento** | `19 Nov – 20 Nov 2026` | Primeras 48 horas (impresiones iniciales sin enfoque técnico). |
| **3. Bugs y Técnica** | `19 Nov 2026 en adelante` | Reportes de `fps`, `crash`, `lag`, `optimización` o `parches`. |
| **4. Opinión Post-Juego** | `21 Nov 2026 en adelante` | Valoración consolidada de historia, mapa y jugabilidad. |

### 🆕 Mejoras incluidas en esta versión (v7)
* **Cuota íntegra en `r/GTA6` + extracción dedicada en español:** Ya no se divide la cuota diaria con `r/GTAVI` (evitando quedarte corto de registros) y se añade una pasada específica en comunidades hispanas (`r/Videojuegos`, `r/esGaming`, `r/Argaming`) filtrando por GTA 6 para habilitar la comparativa `ES` vs `EN` en Power BI.
* **Umbral mínimo de 40 caracteres (`MIN_CARACTERES = 40`):** Evita que `langdetect` confunda frases gamer cortas con otros idiomas y descarte registros válidos.
* **Cálculo estricto de viralidad (`es_viral`):** Exige superar estrictamente el percentil 95 (`>`) y tener `engagement_total > 1`, evitando falsos positivos masivos en comentarios recién publicados con 1 voto.
* **Categoría temática más precisa:** Se renombra `Precio/Hype` a `Hype/Precio/Comparativas` para reflejar correctamente las menciones a tráilers y comparaciones con GTA V.

### 🔑 Antes de ejecutar (fuente: Reddit vía Arctic Shift)
1. **No necesitas credenciales ni editar nada.** La extracción usa **Arctic Shift** (`arctic-shift.photon-reddit.com`), un índice público de Reddit que no requiere cuenta ni API key.
2. Ejecuta todo en orden. La sección 1 descarga los datos y los guarda en `data/raw/reddit_gta6.csv`; si ese archivo ya existe, omite la descarga.

> **Nota:** El dataset conserva `tweet_id` por compatibilidad con el modelo de Power BI, conteniendo el ID del post (`t3_...`) o del comentario (`t1_...`). `likes` = score, `retweets` = 0 y `replies` = número de comentarios del post.

## 📦 Instalación de dependencias
Ejecuta esta celda solo la primera vez y reinicia el kernel después.  
*(Nota: Si usas Python 3.14 y `torch` aún no tiene binarios precompilados, selecciona un entorno con **Python 3.11 o 3.12** en VS Code).* 

In [ ]:
%pip install transformers torch pandas numpy tqdm langdetect requests

## ⚙️ 0. Configuración del Entorno y Parámetros

In [12]:
import os
import re
import glob
import random
from datetime import datetime, timedelta, timezone

import openpyxl
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import time
import requests
from transformers import pipeline
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

DetectorFactory.seed = 0  # resultados reproducibles en langdetect

# ---------------- PARÁMETROS DEL PROYECTO ----------------
TZ_LOCAL = "America/Santo_Domingo"   # zona horaria para analizar las etapas (UTC-4)
FUENTE_EN_UTC = True                 # Reddit/Arctic Shift entregan fechas en UTC
SOLO_ES_EN = True                    # True = conserva únicamente textos en español e inglés
PESOS_ENGAGEMENT = {"likes": 1, "retweets": 2, "replies": 3, "quotes": 2}  # documentar en el dashboard
PERCENTIL_VIRAL = 0.95               # Top 5% de engagement
FECHA_LANZAMIENTO = "2026-11-19"
FECHA_FIN_LANZAMIENTO = "2026-11-20"
# ---------------------------------------------------------

# ---------------- REDDIT vía ARCTIC SHIFT (sin credenciales) ----------------
ARCTIC_URL = "https://arctic-shift.photon-reddit.com/api"
USER_AGENT = "gta6-sentiment-project/1.1"
PAUSA_SEG = 1.0          # pausa entre solicitudes; si ves 429, súbela a 2.0
MAX_PAGINAS_DIA = 10     # tope de páginas (100 filas c/u) por día

# Subreddit principal en inglés (recibe la cuota completa por ventana temporal)
SUBREDDIT_PRINCIPAL_EN = "GTA6"

# Subreddits en español (se filtran por menciones a GTA 6 para balancear ES vs EN)
SUBREDDITS_ES = ["Videojuegos", "esGaming", "Argaming"]
CONSULTAS_ES = ["GTA 6", "GTA VI"]
META_FILAS_ES = 1500     # objetivo adicional de registros en español

OBJETIVO_FILAS_EN = 10500  # objetivo en inglés repartido por ventanas (~12,000 total con ES)
PESO_VENTANAS = {"expectativa": 0.35, "lanzamiento": 0.25, "post": 0.40}  # debe sumar 1.0

# Límites en UTC (Santo Domingo = UTC-4 -> 00:00 local = 04:00 UTC)
INICIO_UTC = datetime(2026, 11, 1, 4, 0, tzinfo=timezone.utc)        # 1 nov 00:00 local
LANZAMIENTO_UTC = datetime(2026, 11, 19, 4, 0, tzinfo=timezone.utc)  # 19 nov 00:00 local
POSTJUEGO_UTC = datetime(2026, 11, 21, 4, 0, tzinfo=timezone.utc)    # 21 nov 00:00 local

MIN_CARACTERES = 40      # evita falsos positivos de langdetect en frases gamer muy cortas
MAX_CARACTERES = 1000    # recorta textos muy largos (el modelo trunca a 128 tokens)

FORZAR_REEXTRACCION = False   # True = vuelve a descargar aunque ya exista data/raw/reddit_gta6.csv
USAR_DATOS_SINTETICOS = False # True = solo para probar el flujo con datos ficticios
# ----------------------------------------------------------------------------

for folder in ["data/raw", "data/interim", "data/processed"]:
    os.makedirs(folder, exist_ok=True)

device = 0 if torch.cuda.is_available() else -1
hardware = torch.cuda.get_device_name(0) if device == 0 else "CPU"
print(f"Entorno listo | Aceleración detectada: {hardware}")

Entorno listo | Aceleración detectada: CPU


## 🟠 1. Extracción de Datos desde Reddit (Arctic Shift)
1. Descarga posts y comentarios de **`r/GTA6` día por día**, respetando una cuota equilibrada por ventana (`Expectativa`, `Lanzamiento`, `Post-juego`).
2. Ejecuta una búsqueda dirigida en **subreddits en español** (`r/Videojuegos`, `r/esGaming`, `r/Argaming`) para capturar conversación hispana sobre GTA 6.
3. Guarda todo en `data/raw/reddit_gta6.csv` (si interrumpes la ejecución con Stop, conserva lo descargado hasta ese momento).

In [2]:
RUTA_REDDIT = "data/raw/reddit_gta6.csv"
FIN_UTC = datetime.now(timezone.utc)

SESION = requests.Session()
SESION.headers.update({"User-Agent": USER_AGENT})


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%S")


def a_fecha_utc(timestamp):
    return datetime.fromtimestamp(timestamp, tz=timezone.utc)


def pedir_json(endpoint, params, reintentos=5):
    """GET a Arctic Shift con reintentos. Devuelve la lista 'data' o None si falla."""
    ultimo_estado = None
    for intento in range(reintentos):
        try:
            r = SESION.get(ARCTIC_URL + endpoint, params=params, timeout=60)
        except requests.RequestException as e:
            ultimo_estado = f"error de red: {e}"
            time.sleep(5 * (intento + 1))
            continue
        if r.status_code == 200:
            time.sleep(PAUSA_SEG)
            try:
                return r.json().get("data", [])
            except ValueError:
                ultimo_estado = "respuesta no es JSON"
                continue
        ultimo_estado = f"HTTP {r.status_code}: {r.text[:150]!r}"
        if r.status_code == 429:
            espera = int(r.headers.get("Retry-After", 30))
            print(f"  [429] límite alcanzado, esperando {espera + 1}s...")
            time.sleep(espera + 1)
        else:
            time.sleep(5 * (intento + 1))
    print(f"  [aviso] {endpoint} sin respuesta -> {ultimo_estado}")
    return None


def es_bot(autor):
    nombre = str(autor).lower() if autor is not None else ""
    return nombre in ("automoderator", "none", "[deleted]") or nombre.endswith("bot")


def texto_valido(texto):
    if not texto:
        return False
    t = texto.strip()
    return t.lower() not in ("[deleted]", "[removed]") and len(t) >= MIN_CARACTERES


def fila_post(p):
    if es_bot(p.get("author")):
        return None
    texto = (p.get("title") or "").strip()
    cuerpo = (p.get("selftext") or "").strip()
    if texto_valido(cuerpo):
        texto = f"{texto}. {cuerpo}"
    if not texto_valido(texto):
        return None
    return {
        "tweet_id": f"t3_{p['id']}",
        "fecha_hora": a_fecha_utc(p["created_utc"]).strftime("%Y-%m-%d %H:%M:%S"),
        "texto_original": texto[:MAX_CARACTERES],
        "likes": max(0, int(p.get("score") or 0)),
        "retweets": 0,
        "replies": int(p.get("num_comments") or 0),
        "quotes": 0,
        "tipo": "post",
        "subreddit": p.get("subreddit", ""),
    }


def fila_comentario(c):
    cuerpo = c.get("body") or ""
    if es_bot(c.get("author")) or not texto_valido(cuerpo):
        return None
    return {
        "tweet_id": f"t1_{c['id']}",
        "fecha_hora": a_fecha_utc(c["created_utc"]).strftime("%Y-%m-%d %H:%M:%S"),
        "texto_original": cuerpo.strip()[:MAX_CARACTERES],
        "likes": max(0, int(c.get("score") or 0)),
        "retweets": 0,
        "replies": 0,
        "quotes": 0,
        "tipo": "comentario",
        "subreddit": c.get("subreddit", ""),
    }


def descargar(endpoint, sub, inicio, fin, max_filas, convertir, extra_params=None):
    """Pagina por fecha (orden ascendente) hasta juntar max_filas válidas o agotar el rango."""
    filas, desde, paginas = [], inicio, 0
    while len(filas) < max_filas and paginas < MAX_PAGINAS_DIA:
        params = {"subreddit": sub, "after": iso(desde), "before": iso(fin), "limit": 100, "sort": "asc"}
        if extra_params:
            params.update(extra_params)
        datos = pedir_json(endpoint, params)
        paginas += 1
        if not datos:
            break
        for d in datos:
            f = convertir(d)
            if f:
                filas.append(f)
        if len(datos) < 100:
            break
        ultimo = a_fecha_utc(datos[-1]["created_utc"])
        desde = max(ultimo, desde + timedelta(seconds=1))
    return filas[:max_filas]


def dias_de_ventana(inicio, fin):
    dias, d = [], inicio
    while d < fin:
        dias.append((d, min(d + timedelta(days=1), fin)))
        d += timedelta(days=1)
    return dias


def extraer_reddit():
    ventanas = [
        ("1. Expectativa", INICIO_UTC, LANZAMIENTO_UTC, PESO_VENTANAS["expectativa"]),
        ("2-3. Lanzamiento", LANZAMIENTO_UTC, POSTJUEGO_UTC, PESO_VENTANAS["lanzamiento"]),
        ("4. Post-juego", POSTJUEGO_UTC, FIN_UTC, PESO_VENTANAS["post"]),
    ]
    filas, vistos = [], set()

    try:
        # 1) Extracción principal en inglés (r/GTA6 con cuota diaria completa)
        for nombre, ini, fin, peso in ventanas:
            dias = dias_de_ventana(ini, fin)
            if not dias:
                continue
            cuota = max(15, int(round(OBJETIVO_FILAS_EN * peso / len(dias))))
            print(f"\n== [EN] r/{SUBREDDIT_PRINCIPAL_EN} | {nombre}: {len(dias)} días | cuota ~{cuota} filas/día")
            for d_ini, d_fin in tqdm(dias, desc=nombre):
                n_dia = 0
                posts = descargar("/posts/search", SUBREDDIT_PRINCIPAL_EN, d_ini, d_fin, max(1, cuota // 5), fila_post)
                coms = descargar("/comments/search", SUBREDDIT_PRINCIPAL_EN, d_ini, d_fin, max(1, cuota - len(posts)), fila_comentario)
                for f in posts + coms:
                    if f["tweet_id"] not in vistos:
                        vistos.add(f["tweet_id"])
                        filas.append(f)
                        n_dia += 1
                print(f"  {d_ini:%Y-%m-%d}: +{n_dia} filas | acumulado {len(filas):,}")

        # 2) Extracción complementaria en español (filtrada por menciones a GTA 6)
        if SUBREDDITS_ES and META_FILAS_ES > 0:
            filas_es = 0
            cuota_sub = max(50, META_FILAS_ES // len(SUBREDDITS_ES))
            print(f"\n== [ES] Comunidades hispanas {SUBREDDITS_ES} | meta ~{META_FILAS_ES} filas")
            for sub in SUBREDDITS_ES:
                for q in CONSULTAS_ES:
                    if filas_es >= META_FILAS_ES:
                        break
                    posts_es = descargar("/posts/search", sub, INICIO_UTC, FIN_UTC, cuota_sub // 3, fila_post, {"query": q})
                    coms_es = descargar("/comments/search", sub, INICIO_UTC, FIN_UTC, cuota_sub, fila_comentario, {"body": q})
                    for f in posts_es + coms_es:
                        if f["tweet_id"] not in vistos:
                            vistos.add(f["tweet_id"])
                            filas.append(f)
                            filas_es += 1
            print(f"  Registros recolectados en comunidades hispanas: +{filas_es:,}")

    except KeyboardInterrupt:
        print("Interrumpido manualmente: se guardará lo recolectado hasta ahora.")

    if not filas:
        raise RuntimeError("No se recolectó nada. Revisa los mensajes [aviso] de arriba (conexión o Arctic Shift).")

    df_reddit = pd.DataFrame(filas).drop_duplicates(subset=["tweet_id"])
    df_reddit.to_csv(RUTA_REDDIT, index=False, encoding="utf-8-sig")

    print(f"\n✅ Guardado: {RUTA_REDDIT} | {len(df_reddit):,} filas únicas")
    print("\nDistribución por tipo:")
    print(df_reddit["tipo"].value_counts().to_string())
    print("\nDistribución por subreddit:")
    print(df_reddit["subreddit"].value_counts().to_string())
    print("\nFilas por día (UTC) - revisa que todas las etapas tengan datos:")
    print(df_reddit.groupby(df_reddit["fecha_hora"].str[:10]).size().to_string())
    return df_reddit


if USAR_DATOS_SINTETICOS:
    print("⚠️ MODO PRUEBA: se omite Reddit y se usarán datos sintéticos en la sección 2.")
elif os.path.exists(RUTA_REDDIT) and not FORZAR_REEXTRACCION:
    print(f"Ya existe {RUTA_REDDIT}: se omite la extracción (usa FORZAR_REEXTRACCION = True para repetirla).")
else:
    extraer_reddit()


== [EN] r/GTA6 | 1. Expectativa: 18 días | cuota ~204 filas/día


1. Expectativa:   6%|▌         | 1/18 [00:09<02:41,  9.51s/it]

  2026-11-01: +204 filas | acumulado 204


1. Expectativa:  11%|█         | 2/18 [00:18<02:30,  9.43s/it]

  2026-11-02: +204 filas | acumulado 408


1. Expectativa:  17%|█▋        | 3/18 [00:28<02:24,  9.66s/it]

... (42 outputs left out)

  2026-11-24: +840 filas | acumulado 8,740


4. Post-juego: 100%|██████████| 5/5 [03:41<00:00, 44.40s/it]

  2026-11-25: +811 filas | acumulado 9,551

== [ES] Comunidades hispanas ['Videojuegos', 'esGaming', 'Argaming'] | meta ~1500 filas


  [aviso] /comments/search sin respuesta -> HTTP 422: '{"data":null,"error":"Timeout. Maybe slow down a bit"}'
  [429] límite alcanzado, esperando 31s...
  [aviso] /posts/search sin respuesta -> HTTP 422: '{"data":null,"error":"Timeout. Maybe slow down a bit"}'
  Registros recolectados en comunidades hispanas: +1,638

✅ Guardado: data/raw/reddit_gta6.csv | 11,189 filas únicas

Distribución por tipo:
tipo
comentario    8868
post          2321

Distribución por subreddit:
subreddit
GTA6           9551
Argaming        926
videojuegos     572
esGaming        140

Filas por día (UTC) - revisa que todas las etapas tengan datos:
fecha_hora
2026-11-01     208
2026-11-02     223
2026-11-03     215
2026-11-04     231
2026-11-05     247
2026-11-06     223
2026-11-07     214
2026-11-08     213
2026-11-09     230
2026-11-10     256
2026-11-11     260
2026-11-12     247
2026-11-13     237
2026-11-14     237
2026-11-15     239
2026-11-16     273
2026-11-17     315
2026-11-18     431
2026-11-19    108

## 📥 2. Ingesta y Unificación de Datos
Lee todos los archivos `.csv` ubicados en `data/raw/` (incluido el de Reddit) y elimina duplicados por `tweet_id`. Si no hay ningún CSV y `USAR_DATOS_SINTETICOS = False`, se detiene con un error claro en vez de analizar datos ficticios.

**Columnas obligatorias:** `tweet_id`, `fecha_hora`, `texto_original`, `likes`, `retweets`, `replies`.  
**Columnas opcionales:** `idioma`, `quotes`, `views`, `username`, `followers`, `subreddit`, `tipo` (post/comentario).

In [5]:
def generar_datos_prueba(n_filas: int = 12000) -> pd.DataFrame:
    """Genera un dataset bilingüe sintético cubriendo las 4 etapas de GTA 6."""
    random.seed(42)
    np.random.seed(42)

    plantillas = [
        # Etapa 1: Expectativa (1 - 18 nov)
        ("es", "pre", "El hype por GTA 6 está en otro nivel, ya quiero explorar Vice City 🔥", 120, 30, 12),
        ("en", "pre", "Rockstar never misses. Counting down the days for GTA VI release!", 340, 85, 24),
        ("es", "pre", "El precio de GTA 6 está bastante alto pero igual va a valer cada centavo", 95, 14, 45),
        ("en", "pre", "Comparing GTA V vs GTA 6 trailers, the lighting and graphics leap is insane", 410, 92, 31),
        ("es", "pre", "Faltan pocos días para GTA 6 y no puedo con la ansiedad, ya reservé mi copia", 180, 40, 20),
        ("es", "pre", "Otro tráiler de GTA VI y yo sin dormir, Rockstar nos tiene en vilo", 260, 55, 18),
        ("es", "pre", "No sé si GTA 6 valga ese precio, está carísimo para mi bolsillo", 75, 10, 52),
        ("es", "pre", "Mi PC no va a poder con GTA 6, toca ahorrar para una consola nueva", 60, 8, 30),
        ("en", "pre", "Preordered GTA VI today, the hype is unreal and I cannot wait", 300, 70, 22),
        ("en", "pre", "Is GTA 6 really worth the price? $70 feels like a lot for base edition", 150, 25, 68),
        ("en", "pre", "Rockstar delaying again would be a disaster, please no more waiting", 220, 48, 55),
        ("en", "pre", "The new trailer shows Vice City like never before, graphics look unreal", 520, 130, 35),
        # Etapa 2: Lanzamiento (19 - 20 nov)
        ("es", "launch", "Ya abriendo GTA 6! La intro con Jason y Lucia empieza durísima", 520, 110, 40),
        ("en", "launch", "GTA 6 is finally live! Leonida looks breathtaking on console", 890, 210, 65),
        ("es", "launch", "Llevo 3 horas jugando GTA VI y la atmósfera de Vice City es una locura", 230, 45, 19),
        ("es", "launch", "GTA 6 ya está disponible! Descargando ahora mismo, qué emoción", 480, 100, 36),
        ("es", "launch", "Primeras impresiones de GTA VI: el mundo abierto se ve gigantesco", 350, 70, 28),
        ("es", "launch", "Ya probé GTA 6 y el mapa de Leonida es enorme, no sé por dónde empezar", 290, 58, 24),
        ("en", "launch", "Just launched GTA VI and the opening mission is an absolute banger", 760, 175, 58),
        ("en", "launch", "First hour of GTA 6 done, Jason and Lucia have great chemistry", 640, 140, 47),
        ("en", "launch", "Downloading GTA VI right now, 150GB is no joke on launch night", 410, 85, 39),
        # Etapa 3: Bugs y técnica (19 nov en adelante)
        ("en", "bugs", "Severe frame drops and fps stutter in downtown Vice City, needs a patch ASAP", 640, 180, 95),
        ("es", "bugs", "Me dio crash dos veces en la misma misión, pésima optimización de salida", 310, 75, 60),
        ("en", "bugs", "Textures not loading properly and audio lag bug after 2 hours of gameplay", 280, 62, 44),
        ("es", "bugs", "GTA 6 se me congela en pantalla de carga, ¿alguien más con el mismo error?", 240, 50, 72),
        ("es", "bugs", "Los fps caen a 20 en la ciudad, necesitan un parche urgente", 380, 95, 66),
        ("es", "bugs", "Se me cierra el juego al azar, crash constante en la misión del banco", 200, 42, 58),
        ("en", "bugs", "GTA VI crashes every time I enter a building, this is unacceptable", 520, 140, 88),
        ("en", "bugs", "Game keeps freezing on the loading screen, anyone else with this error?", 330, 76, 81),
        ("en", "bugs", "Performance on base console is rough, constant lag in heavy traffic", 410, 98, 63),
        # Etapa 4: Opinión consolidada tras jugar (21 nov en adelante)
        ("es", "post", "Terminé el acto 2 de la historia de GTA 6, el guion de Lucia es obra maestra", 760, 150, 52),
        ("en", "post", "After 25 hours in GTA VI, the open world depth and AI reactions beat everything", 1120, 290, 88),
        ("es", "post", "El nuevo parche mejoró bastante el rendimiento y la jugabilidad se siente súper fluida", 430, 88, 27),
        ("en", "post", "Honestly expected more from the side missions, main story is great though", 190, 22, 49),
        ("es", "post", "Después de 30 horas, GTA VI es el mejor juego de Rockstar, sin dudas", 980, 210, 70),
        ("es", "post", "La historia de Jason y Lucia me tiene enganchado, final increíble", 690, 140, 45),
        ("es", "post", "Las misiones secundarias de GTA 6 se sienten repetitivas, esperaba más", 170, 20, 62),
        ("es", "post", "Los gráficos de GTA VI en 4K son una obra de arte en cada esquina", 840, 175, 40),
        ("en", "post", "GTA VI story is a masterpiece, Lucia is the best protagonist in the series", 1050, 260, 79),
        ("en", "post", "The map of Leonida is huge and full of life, best open world ever", 910, 215, 66),
        ("en", "post", "Too many micro-transactions in GTA 6 online, really disappointing", 260, 45, 92),
        ("en", "post", "Graphics are stunning but the side content feels shallow after chapter 3", 230, 38, 57),
    ]

    sufijos = ["", "", "", " #GTA6", " #GTAVI #Rockstar", " https://t.co/xyz123", " @RockstarGames", " #GTA6 🔥"]

    registros = []
    inicio_pre = datetime(2026, 11, 1)

    for i in range(n_filas):
        r = random.random()
        if r < 0.35:
            fecha_dt = inicio_pre + timedelta(days=random.randint(0, 17), hours=random.randint(0, 23), minutes=random.randint(0, 59))
            pool = [p for p in plantillas if p[1] == "pre"]
        elif r < 0.60:
            fecha_dt = datetime(2026, 11, 19) + timedelta(days=random.randint(0, 1), hours=random.randint(0, 23), minutes=random.randint(0, 59))
            pool = [p for p in plantillas if p[1] in ("launch", "bugs")]
        else:
            fecha_dt = datetime(2026, 11, 21) + timedelta(days=random.randint(0, 4), hours=random.randint(0, 23), minutes=random.randint(0, 59))
            pool = [p for p in plantillas if p[1] in ("post", "bugs")]

        idioma, _, texto, base_l, base_rt, base_rep = random.choice(pool)
        texto = texto + random.choice(sufijos)
        factor = np.random.lognormal(mean=0.0, sigma=0.8)

        registros.append({
            "tweet_id": str(185800000000 + i),
            "fecha_hora": fecha_dt.strftime("%Y-%m-%d %H:%M:%S"),
            "idioma": idioma,
            "texto_original": texto,
            "likes": int(base_l * factor),
            "retweets": int(base_rt * factor),
            "replies": int(base_rep * factor),
        })

    return pd.DataFrame(registros)


archivos_raw = sorted(glob.glob("data/raw/*.csv"))
usa_datos_reales = len(archivos_raw) > 0

if usa_datos_reales:
    print(f"Cargando {len(archivos_raw)} archivo(s) desde data/raw/: {[os.path.basename(f) for f in archivos_raw]}")
    df = pd.concat([pd.read_csv(f, dtype={"tweet_id": str}) for f in archivos_raw], ignore_index=True)
elif USAR_DATOS_SINTETICOS:
    print("⚠️ MODO PRUEBA: generando 12,000 filas SINTÉTICAS (ya en hora local). No uses estos resultados para conclusiones.")
    df = generar_datos_prueba(12000)
else:
    raise FileNotFoundError(
        "No hay CSV en data/raw/. Ejecuta la sección 1 (extracción de Reddit vía Arctic Shift), "
        "o pon USAR_DATOS_SINTETICOS = True solo para probar el flujo."
    )

# Validar columnas obligatorias
obligatorias = ["tweet_id", "fecha_hora", "texto_original", "likes", "retweets", "replies"]
faltantes = [c for c in obligatorias if c not in df.columns]
if faltantes:
    raise ValueError(f"Faltan columnas obligatorias en los CSV: {faltantes}")

# Deduplicación
n_inicial = len(df)
df = df.drop_duplicates(subset=["tweet_id"]).copy()

# Fechas: conversión de UTC a hora local (solo datos reales) para que la etapa no se desfase
convertir_tz = usa_datos_reales and FUENTE_EN_UTC
df["fecha_hora"] = pd.to_datetime(df["fecha_hora"], errors="coerce", utc=convertir_tz)
if convertir_tz:
    df["fecha_hora"] = df["fecha_hora"].dt.tz_convert(TZ_LOCAL).dt.tz_localize(None)

df = df.dropna(subset=["fecha_hora", "texto_original"]).copy()
df["fecha"] = df["fecha_hora"].dt.strftime("%Y-%m-%d")
df["hora"] = df["fecha_hora"].dt.hour

# Métricas numéricas: asegurar tipo numérico y valores nulos en 0
for col in ["likes", "retweets", "replies", "quotes"]:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

df.to_csv("data/interim/tweets_unificados.csv", index=False, encoding="utf-8-sig")
print(f"Registros leídos: {n_inicial:,} | Únicos y válidos: {len(df):,}")
df.head(3)

Cargando 1 archivo(s) desde data/raw/: ['reddit_gta6.csv']
Registros leídos: 11,189 | Únicos y válidos: 11,189


,tweet_id,fecha_hora,texto_original,likes,retweets,replies,quotes,tipo,subreddit,fecha,hora
0,t3_1xktnow,2026-11-01 00:06:06,We are so close. Stay safe everyone!. I’m awar...,853,0,31,0,post,GTA6,2026-11-01,0
1,t3_1xktr9y,2026-11-01 00:11:42,Finally did it.. Last console I owned was the ...,36,0,31,0,post,GTA6,2026-11-01,0
2,t3_1xktzxn,2026-11-01 00:25:46,How it’s gonna feel deleting your old games to...,507,0,126,0,post,GTA6,2026-11-01,0


## 🧹 3. Preprocesamiento, Idioma y Filtros de Calidad
Conservamos `texto_original` para mostrar citas reales en los *tooltips* de Power BI y creamos `texto_limpio` (sin URLs ni menciones) para alimentar el transformer y las nubes de palabras.  
Además, extraemos `hashtags`, eliminamos textos que quedaron vacíos tras limpiar y detectamos el idioma (`es` / `en`).

In [6]:
def limpiar_texto(texto: str) -> str:
    texto = str(texto)
    texto = re.sub(r"http\S+|www\.\S+", "", texto)  # Eliminar enlaces
    texto = re.sub(r"@\w+", "", texto)                # Eliminar menciones
    texto = texto.replace("#", "")                    # Conservar texto del hashtag
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def detectar_idioma(texto: str) -> str:
    try:
        return detect(texto)
    except LangDetectException:
        return "und"


# Hashtags (se extraen del texto original, antes de limpiar)
df["hashtags"] = (
    df["texto_original"].astype(str)
    .str.findall(r"#\w+")
    .apply(lambda lista: ",".join(h.lower() for h in lista))
)

df["texto_limpio"] = df["texto_original"].apply(limpiar_texto)

# Filtro: eliminar textos que quedaron vacíos o demasiado cortos tras la limpieza
antes = len(df)
df = df[df["texto_limpio"].str.len() >= 15].copy()
print(f"Textos vacíos o muy cortos eliminados tras limpiar: {antes - len(df):,}")

# Idioma: usar la columna existente y recalcular solo donde falta o es inválida
if "idioma" not in df.columns:
    df["idioma"] = np.nan
df["idioma"] = df["idioma"].astype(str).str.lower().str.strip()
sin_idioma = ~df["idioma"].str.match(r"^[a-z]{2,3}$") | df["idioma"].isin(["und", "nan", "none", ""])
print(f"Registros sin idioma válido (se detectarán con langdetect): {sin_idioma.sum():,}")
df.loc[sin_idioma, "idioma"] = df.loc[sin_idioma, "texto_limpio"].apply(detectar_idioma)

if SOLO_ES_EN:
    antes = len(df)
    df = df[df["idioma"].isin(["es", "en"])].copy()
    print(f"Registros en otros idiomas descartados: {antes - len(df):,}")

df = df.reset_index(drop=True)
df[["idioma", "texto_original", "texto_limpio", "hashtags"]].head(5)

Textos vacíos o muy cortos eliminados tras limpiar: 28
Registros sin idioma válido (se detectarán con langdetect): 11,161
Registros en otros idiomas descartados: 128


,idioma,texto_original,texto_limpio,hashtags
0,en,We are so close. Stay safe everyone!. I’m awar...,We are so close. Stay safe everyone!. I’m awar...,
1,en,Finally did it.. Last console I owned was the ...,Finally did it.. Last console I owned was the ...,
2,en,How it’s gonna feel deleting your old games to...,How it’s gonna feel deleting your old games to...,
3,en,"We're so close, team.. We're almost there guys...","We're so close, team.. We're almost there guys...",
4,en,GTA 6 Online Custom Jobs. Might be a little to...,GTA 6 Online Custom Jobs. Might be a little to...,


## 🧠 4. Análisis de Sentimiento con NLP (`XLM-RoBERTa`)
Ejecutamos inferencia por lotes usando `cardiffnlp/twitter-xlm-roberta-base-sentiment`, generando:
* `sentimiento`: `Positivo`, `Neutral`, `Negativo`
* `score_sentimiento`: `1`, `0`, `-1` (ideal para calcular el **Net Sentiment Score** en DAX)
* `confianza_nlp`: probabilidad del modelo (`0.00` a `1.00`)

In [7]:
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
modelo_nlp = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=device,
    truncation=True,
    max_length=128
)

textos = df["texto_limpio"].tolist()
batch_size = 64 if device == 0 else 32
resultados = []

for i in tqdm(range(0, len(textos), batch_size), desc="Clasificando sentimiento"):
    resultados.extend(modelo_nlp(textos[i : i + batch_size]))

mapa_etiquetas = {
    "positive": ("Positivo", 1),
    "neutral": ("Neutral", 0),
    "negative": ("Negativo", -1),
}

df["sentimiento"] = [mapa_etiquetas.get(r["label"].lower(), ("Neutral", 0))[0] for r in resultados]
df["score_sentimiento"] = [mapa_etiquetas.get(r["label"].lower(), ("Neutral", 0))[1] for r in resultados]
df["confianza_nlp"] = [round(float(r["score"]), 4) for r in resultados]

df[["texto_limpio", "sentimiento", "score_sentimiento", "confianza_nlp"]].head(5)

Clasificando sentimiento: 100%|██████████| 345/345 [15:36<00:00,  2.71s/it]


,texto_limpio,sentimiento,score_sentimiento,confianza_nlp
0,We are so close. Stay safe everyone!. I’m awar...,Positivo,1,0.4244
1,Finally did it.. Last console I owned was the ...,Positivo,1,0.5991
2,How it’s gonna feel deleting your old games to...,Neutral,0,0.5483
3,"We're so close, team.. We're almost there guys...",Positivo,1,0.7491
4,GTA 6 Online Custom Jobs. Might be a little to...,Negativo,-1,0.5634


## 🏷️ 5. Clasificación de Temas, 4 Etapas y Engagement
Combinamos reglas de palabras clave bilingües (**con límites de palabra `\b`**) con la fecha de publicación para separar el ruido técnico (`3. Bugs y Técnica`) de la experiencia general (`4. Opinión Post-Juego`).

La viralidad (`es_viral` y `es_viral_global`) exige superar estrictamente el percentil 95 (`>`) y contar con más de 1 punto de engagement para evitar falsos positivos en comentarios recientes.

In [8]:
KW_BUGS = ["bug", "bugs", "crash", "crashes", "crasheo", "fps", "lag", "laggy", "stutter", "freeze",
           "optimizacion", "optimización", "parche", "patch", "error", "fallo", "rendimiento", "performance"]
KW_HISTORIA = ["historia", "story", "guion", "lucia", "jason", "personaje", "personajes", "character", "characters",
               "mision", "misión", "misiones", "mission", "missions", "final", "ending"]
KW_GRAFICOS = ["grafico", "gráfico", "graficos", "gráficos", "graphics", "iluminacion", "iluminación", "lighting",
               "textura", "texturas", "texture", "textures", "mapa", "map", "vice city", "leonida", "4k"]
KW_HYPE_PRECIO = ["precio", "price", "caro", "expensive", "hype", "trailer", "trailers", "tráiler", "reservar",
                  "preorder", "gta v", "gta 5"]


def compilar(palabras):
    """Compila un regex con límites de palabra: 'final' ya no coincide con 'finally'."""
    patron = r"\b(?:" + "|".join(re.escape(p) for p in palabras) + r")\b"
    return re.compile(patron, flags=re.IGNORECASE)


RE_BUGS = compilar(KW_BUGS)
RE_HISTORIA = compilar(KW_HISTORIA)
RE_GRAFICOS = compilar(KW_GRAFICOS)
RE_HYPE_PRECIO = compilar(KW_HYPE_PRECIO)


def detectar_tema(texto: str) -> str:
    if RE_BUGS.search(texto):
        return "Bugs/Rendimiento"
    if RE_HISTORIA.search(texto):
        return "Historia/Personajes"
    if RE_GRAFICOS.search(texto):
        return "Gráficos"
    if RE_HYPE_PRECIO.search(texto):
        return "Hype/Precio/Comparativas"
    return "General"


df["tema_principal"] = df["texto_limpio"].apply(detectar_tema)

# Reglas de asignación de las 4 Etapas (sobre fecha ya en hora local)
condiciones = [
    df["fecha"] < FECHA_LANZAMIENTO,
    (df["fecha"] >= FECHA_LANZAMIENTO) & (df["tema_principal"] == "Bugs/Rendimiento"),
    (df["fecha"] >= FECHA_LANZAMIENTO) & (df["fecha"] <= FECHA_FIN_LANZAMIENTO),
]
opciones_etapa = [
    "1. Expectativa",
    "3. Bugs y Técnica",
    "2. Lanzamiento",
]
df["etapa"] = np.select(condiciones, opciones_etapa, default="4. Opinión Post-Juego")

# Engagement ponderado (pesos configurables arriba)
df["engagement_total"] = sum(df[col] * peso for col, peso in PESOS_ENGAGEMENT.items())

# Viralidad global y por etapa (estrictamente > percentil 95 y > 1 punto para evitar empates en score=1)
umbral_global = df["engagement_total"].quantile(PERCENTIL_VIRAL)
df["es_viral_global"] = ((df["engagement_total"] > umbral_global) & (df["engagement_total"] > 1)).astype(int)

umbral_por_etapa = df.groupby("etapa")["engagement_total"].transform(lambda s: s.quantile(PERCENTIL_VIRAL))
df["es_viral"] = ((df["engagement_total"] > umbral_por_etapa) & (df["engagement_total"] > 1)).astype(int)

print(f"Umbral viral global (Top 5%): > {umbral_global:,.0f} puntos de engagement")
print("\nUmbral viral por etapa:")
print(df.groupby("etapa")["engagement_total"].quantile(PERCENTIL_VIRAL).round(0).to_string())

Umbral viral global (Top 5%): > 59 puntos de engagement

Umbral viral por etapa:
etapa
1. Expectativa           105.0
2. Lanzamiento            44.0
3. Bugs y Técnica        104.0
4. Opinión Post-Juego     33.0


## 📊 6. Auditoría de Resultados y Exportación del Dataset Maestro (`gta6_master.csv`)

In [13]:
columnas_base = [
    "tweet_id", "fecha_hora", "fecha", "hora", "idioma",
    "texto_original", "texto_limpio", "hashtags",
    "likes", "retweets", "replies", "quotes", "engagement_total",
    "sentimiento", "score_sentimiento", "confianza_nlp", "tema_principal",
    "etapa", "es_viral", "es_viral_global"
]
columnas_opcionales = [c for c in ["username", "followers", "views", "subreddit", "tipo"] if c in df.columns]

columnas_finales = columnas_base + columnas_opcionales
df_final = df[columnas_finales].copy()

# Eliminar caracteres de control invisibles de Reddit que rompen el formato XML de Excel
for col in ["texto_original", "texto_limpio", "hashtags"]:
    if col in df_final.columns:
        df_final[col] = (
            df_final[col]
            .fillna("")
            .astype(str)
            .apply(lambda s: re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", s))
        )

# Tablas resumen por Etapa
resumen = df_final.groupby("etapa").agg(
    total_registros=("tweet_id", "count"),
    net_sentiment_score=("score_sentimiento", "mean"),
    engagement_promedio=("engagement_total", "mean"),
    registros_virales=("es_viral", "sum")
).round(2)

tabla_sentimiento = pd.crosstab(df_final["etapa"], df_final["sentimiento"], normalize="index").mul(100).round(1)
tabla_temas = pd.crosstab(df_final["etapa"], df_final["tema_principal"], normalize="index").mul(100).round(1)

# Exportar a Excel (.xlsx) con 4 hojas organizadas + respaldo CSV
ruta_xlsx = "data/processed/gta6_master.xlsx"
ruta_csv = "data/processed/gta6_master.csv"

with pd.ExcelWriter(ruta_xlsx, engine="openpyxl") as writer:
    df_final.to_excel(writer, sheet_name="Datos_Maestros", index=False)
    resumen.to_excel(writer, sheet_name="Resumen_Etapas")
    tabla_sentimiento.to_excel(writer, sheet_name="Sentimiento_Por_Etapa")
    tabla_temas.to_excel(writer, sheet_name="Temas_Por_Etapa")

df_final.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

print(f"✅ Dataset maestro exportado en Excel ({len(df_final):,} filas y {len(df_final.columns)} columnas) -> {ruta_xlsx}")
print(f"✅ Respaldo CSV exportado -> {ruta_csv}")
print(f"Columnas opcionales incluidas: {columnas_opcionales if columnas_opcionales else 'ninguna'}")
print(f"Distribución por idioma: {df_final['idioma'].value_counts().to_dict()}\n")

display(resumen)

print("Distribución de sentimiento por etapa (%):")
display(tabla_sentimiento)

print("Distribución de temas por etapa (%):")
display(tabla_temas)

✅ Dataset maestro exportado en Excel (11,033 filas y 22 columnas) -> data/processed/gta6_master.xlsx
✅ Respaldo CSV exportado -> data/processed/gta6_master.csv
Columnas opcionales incluidas: ['subreddit', 'tipo']
Distribución por idioma: {'en': 9442, 'es': 1591}



,total_registros,net_sentiment_score,engagement_promedio,registros_virales
etapa,,,,
1. Expectativa,4477,-0.21,37.87,221
2. Lanzamiento,1939,-0.29,21.34,97
3. Bugs y Técnica,278,-0.57,28.77,14
4. Opinión Post-Juego,4339,-0.27,13.51,211


Distribución de sentimiento por etapa (%):


sentimiento,Negativo,Neutral,Positivo
etapa,,,
1. Expectativa,42.9,35.2,21.9
2. Lanzamiento,49.1,30.3,20.5
3. Bugs y Técnica,69.8,17.6,12.6
4. Opinión Post-Juego,47.9,31.6,20.6


Distribución de temas por etapa (%):


tema_principal,Bugs/Rendimiento,General,Gráficos,Historia/Personajes,Hype/Precio/Comparativas
etapa,,,,,
1. Expectativa,4.0,76.0,4.3,9.7,6.0
2. Lanzamiento,0.0,76.7,4.4,15.2,3.6
3. Bugs y Técnica,100.0,0.0,0.0,0.0,0.0
4. Opinión Post-Juego,0.0,73.2,3.5,20.7,2.6
